# 365 Probabilidades - Dia #121
## Qual a probabilidade de uma criança passar cinco minutos sem fazer uma pergunta?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-12
**Ferramenta:** Python
**Decisão analisada:** Quanto tempo de silêncio dá para esperar de uma criança numa conversa?
**Hashtag:** #365Probabilidades #Dia121

---

### 📖 A História

[Abertura da Ana. Uma lembrança ou cena dela com perguntas de criança, a inserir.]

Hoje é Dia das Crianças. Fui procurar se alguém já tinha contado quantas perguntas uma
criança faz.

---

### 📚 O Conceito: perguntas como chegadas

Uma pesquisadora, Michelle Chouinard, contou. Ela leu transcrições de conversas de
crianças pequenas com adultos conhecidos e registrou cada pergunta. O ritmo que ficou
conhecido é de 76 perguntas de busca de informação por hora de conversa.

Quando um evento acontece muitas vezes, num ritmo médio conhecido, a estatística tem um
modelo pronto para a pergunta "qual a chance de não acontecer nenhum em tanto tempo?".
É o processo de Poisson. Se as perguntas chegam a um ritmo λ por minuto, a chance de
passar t minutos sem nenhuma é

$$P(\text{nenhuma pergunta em } t \text{ minutos}) = e^{-\lambda t}$$

O modelo supõe que as perguntas chegam de forma independente, no mesmo ritmo o tempo
todo. Criança não funciona assim. Perguntas vêm em rajadas, e uma resposta puxa outra
pergunta. Por isso a conta tem duas versões: a de Poisson, que é a referência, e uma com
rajadas, que mostra para que lado a resposta se move.

---

### 🧮 O Modelo

Cinco camadas. A fonte dá o ritmo. As contas são nossas.

1. **O silêncio.** A chance de 1, 2, 5 e 10 minutos sem pergunta, pela equação acima,
   com o ritmo de 76 por hora e com a faixa de uma a três perguntas por minuto.
2. **A espera.** Quanto tempo até a próxima pergunta, e quanto tempo de conversa é
   preciso, em média, até aparecer o primeiro silêncio de cinco minutos. A fórmula é
   conferida por simulação.
3. **As rajadas.** A mesma conta com o ritmo variando de um trecho da conversa para
   outro (mistura Gama-Poisson). O grau de rajada não é publicado, então é varrido.
4. **Uma tarde.** Monte Carlo de três horas de conversa: quantas perguntas, o maior
   silêncio e a chance de haver pelo menos um silêncio de cinco minutos.
5. **Até os cinco anos.** Conta de bolso dos pedidos de explicação acumulados.

**Fonte:**
- Chouinard, M. M. (2007). "Children's Questions: A Mechanism for Cognitive
  Development". *Monographs of the Society for Research in Child Development* 72(1),
  1-129. Estudo 1: transcrições de **quatro crianças**, de 1 ano e 2 meses a 5 anos e
  1 mês, em conversa com adultos conhecidos (base CHILDES). Estudo 2: diário de
  perguntas mantido por **68 pais**, crianças de 1 a 5 anos. Estudo 3: 112 duplas de
  pais e filhos em zoológicos. Estudo 4: 67 crianças de 4 anos. **Desenho lido no
  resumo oficial. A monografia completa não foi lida.**

**De onde vêm os ritmos (fontes secundárias, declaradas):**
- 76 perguntas de busca de informação por hora: Kurkul, K. E., Dwyer, J. & Corriveau,
  K. H. (2022), *Early Childhood Research Quarterly* 58, 231-241, e Vlach, H. A. &
  Lazaroff, E. (2022), *Frontiers in Psychology* 13, 892298, os dois citando Chouinard
  (2007).
- De uma a três perguntas por minuto, cerca de dois terços delas de busca de
  informação, e cerca de 25 pedidos de explicação por hora: Harris, P. L. (2015), "What
  Children Learn from Questioning", *Educational Leadership*, setembro de 2015, resumindo o Estudo 1.

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. Ele vale para proporções
de pesquisa de opinião autorrelatada, e aqui a medida é contagem de perguntas em
transcrição.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 200_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---
# Chouinard (2007), Monographs of the SRCD 72(1). Ritmos lidos em fontes secundárias.

# Perguntas de busca de informação por hora de conversa com um adulto conhecido
# (Kurkul, Dwyer & Corriveau 2022; Vlach & Lazaroff 2022, citando Chouinard 2007).
PERGUNTAS_HORA = 76
LAMBDA = PERGUNTAS_HORA / 60          # perguntas por minuto

# Todas as perguntas, de qualquer tipo: de uma a três por minuto (Harris 2015).
FAIXA_TOTAL = (1.0, 3.0)              # perguntas por minuto
FRACAO_INFORMACAO = 2 / 3             # "cerca de dois terços" buscam informação (Harris 2015)

# Pedidos de explicação ("por quê?", "como?") por hora (Harris 2015).
EXPLICACOES_HORA = 25

# Amostra do Estudo 1 (resumo oficial da monografia).
N_CRIANCAS = 4

MINUTOS = 5                            # o silêncio da pergunta do dia

print("=" * 70)
print("  DADOS · CHOUINARD (2007)")
print("=" * 70)
print(f"\n  Estudo 1: {N_CRIANCAS} crianças, de 1 ano e 2 meses a 5 anos e 1 mês")
print(f"  Perguntas de busca de informação: {PERGUNTAS_HORA} por hora de conversa")
print(f"  Ritmo por minuto (λ): {vir(LAMBDA, 2)} · uma pergunta a cada {vir(60 / LAMBDA, 0)} segundos, em média")
print(f"  Todas as perguntas: de {vir(FAIXA_TOTAL[0], 0)} a {vir(FAIXA_TOTAL[1], 0)} por minuto")
print(f"  Conferência: {PERGUNTAS_HORA} por hora de busca de informação, se forem dois terços do "
      f"total, dão {vir(PERGUNTAS_HORA / FRACAO_INFORMACAO, 0)} perguntas por hora, "
      f"ou {vir(PERGUNTAS_HORA / FRACAO_INFORMACAO / 60, 1)} por minuto (dentro da faixa)")
print(f"  Pedidos de explicação: {EXPLICACOES_HORA} por hora")
print("\n  Ritmos lidos em fonte secundária. A monografia completa não foi lida.")
print("=" * 70)

In [ ]:
# --- O MODELO ---
rng = np.random.default_rng(SEMENTE)

def p_silencio(lam, t):
    # Processo de Poisson: chance de nenhuma pergunta em t minutos.
    return np.exp(-lam * t)

# ── CAMADA 1 · o silêncio ──
p5 = p_silencio(LAMBDA, MINUTOS)
p5_faixa = (p_silencio(FAIXA_TOTAL[1], MINUTOS), p_silencio(FAIXA_TOTAL[0], MINUTOS))
print("=" * 70)
print("  CAMADA 1 · A CHANCE DE SILÊNCIO (processo de Poisson)")
print("=" * 70)
print(f"\n  P(nenhuma pergunta em t minutos) = e^(−λt), com λ = {vir(LAMBDA, 2)} por minuto\n")
for t in (0.5, 1, 2, 5, 10):
    p = p_silencio(LAMBDA, t)
    rotulo = "30 segundos" if t == 0.5 else f"{int(t)} minuto{'s' if t > 1 else ''}"
    casas = 1 if p >= 0.01 else (2 if p >= 0.0001 else 4)
    print(f"    {rotulo:>11}: {vir(p * 100, casas)}%")
print(f"\n  Cinco minutos sem pergunta: {vir(p5 * 100, 2)}% · 1 em {mil(1 / p5)}")
print(f"  Perguntas esperadas em cinco minutos: {vir(LAMBDA * MINUTOS, 1)}")
print(f"\n  Contando todas as perguntas (de 1 a 3 por minuto), cinco minutos de silêncio:")
print(f"    com 1 por minuto: {vir(p5_faixa[1] * 100, 2)}% · com 3 por minuto: "
      f"{vir(p5_faixa[0] * 100, 5)}%")

# ── CAMADA 2 · a espera ──
mediana_seg = np.log(2) / LAMBDA * 60
p90_seg = -np.log(0.10) / LAMBDA * 60
# Tempo médio de conversa até completar o primeiro silêncio de t minutos:
#   E[T] = (e^(λt) − 1) / λ
espera_min = (np.exp(LAMBDA * MINUTOS) - 1) / LAMBDA

# Conferência por simulação: 5.000 conversas longas, intervalo a intervalo.
N_CONF, N_INTERVALOS = 5_000, 4_000
intervalos = rng.exponential(1 / LAMBDA, size=(N_CONF, N_INTERVALOS)).astype(np.float32)
longo = intervalos >= MINUTOS
tem = longo.any(axis=1)
primeiro = longo.argmax(axis=1)
antes = np.where(np.arange(N_INTERVALOS) < primeiro[:, None], intervalos, 0).sum(axis=1)
espera_sim = (antes + MINUTOS)[tem]

print("\n" + "=" * 70)
print("  CAMADA 2 · A ESPERA")
print("=" * 70)
print(f"\n  Tempo até a próxima pergunta: mediana de {vir(mediana_seg, 0)} segundos · "
      f"em 90% das vezes ela chega em até {vir(p90_seg, 0)} segundos")
print(f"  Tempo médio de conversa até o primeiro silêncio de {MINUTOS} minutos:")
print(f"    pela fórmula (e^(λt) − 1) / λ: {vir(espera_min, 0)} minutos, ou {vir(espera_min / 60, 1)} horas")
print(f"    pela simulação de {mil(tem.sum())} conversas: {vir(espera_sim.mean(), 0)} minutos, "
      f"ou {vir(espera_sim.mean() / 60, 1)} horas")
assert tem.mean() > 0.99 and abs(espera_sim.mean() / espera_min - 1) < 0.05

# ── CAMADA 3 · as rajadas ──
# O ritmo de cada trecho de conversa varia: λ_trecho ~ Gama(forma k, média λ).
# Então P(nenhuma pergunta em t) = (1 + λt/k)^(−k). Com k muito grande volta a Poisson.
# O coeficiente de variação do ritmo entre trechos é 1/√k.
def p_silencio_rajada(lam, t, k):
    return (1 + lam * t / k) ** (-k)

FORMAS = (np.inf, 20, 5, 2, 1)
print("\n" + "=" * 70)
print("  CAMADA 3 · SE AS PERGUNTAS VÊM EM RAJADAS")
print("=" * 70)
print(f"\n  Mesmo ritmo médio ({PERGUNTAS_HORA} por hora), ritmo variando entre trechos da conversa.")
print("  O grau de rajada não é publicado. Varredura:\n")
rajadas = {}
for k in FORMAS:
    p = p5 if np.isinf(k) else p_silencio_rajada(LAMBDA, MINUTOS, k)
    rajadas[k] = p
    cv = 0 if np.isinf(k) else 1 / np.sqrt(k)
    nome = "sem rajadas (Poisson)" if np.isinf(k) else f"variação de {vir(cv * 100, 0)}% no ritmo"
    print(f"    {nome:<28} cinco minutos de silêncio: {vir(p * 100, 2 if p < 0.01 else 1)}%")
print(f"\n  A resposta vai de {vir(p5 * 100, 2)}% a {vir(rajadas[1] * 100, 1)}% conforme o grau de rajada.")
print("  A direção é uma só: rajadas aumentam a chance de silêncio.")

# ── CAMADA 4 · uma tarde de conversa ──
HORAS = 3
T_TOTAL = HORAS * 60
MAX_EVENTOS = 420
maior_silencio = np.empty(SORTEIOS)
n_perguntas = np.empty(SORTEIOS, dtype=int)
LOTE = 20_000
for ini in range(0, SORTEIOS, LOTE):
    gaps = rng.exponential(1 / LAMBDA, size=(LOTE, MAX_EVENTOS))
    tempos = gaps.cumsum(axis=1)
    dentro = tempos <= T_TOTAL
    assert (~dentro[:, -1]).all()
    n = dentro.sum(axis=1)
    # silêncios: intervalos entre perguntas dentro da tarde, mais o trecho final até o fim
    gaps_dentro = np.where(dentro, gaps, 0)
    ultimo = np.where(n > 0, tempos[np.arange(LOTE), np.maximum(n - 1, 0)], 0)
    final = T_TOTAL - ultimo
    maior_silencio[ini:ini + LOTE] = np.maximum(gaps_dentro.max(axis=1), final)
    n_perguntas[ini:ini + LOTE] = n
np_lo, np_hi = np.percentile(n_perguntas, [2.5, 97.5])
ms_lo, ms_med, ms_hi = np.percentile(maior_silencio, [2.5, 50, 97.5])
p_tarde = (maior_silencio >= MINUTOS).mean()

print("\n" + "=" * 70)
print(f"  CAMADA 4 · UMA TARDE: {HORAS} HORAS DE CONVERSA ({mil(SORTEIOS)} sorteios)")
print("=" * 70)
print(f"\n  Perguntas na tarde: média {vir(n_perguntas.mean(), 0)} · 95% entre {int(np_lo)} e {int(np_hi)}")
print(f"  Maior silêncio da tarde: mediana de {vir(ms_med, 1)} minutos · "
      f"95% entre {vir(ms_lo, 1)} e {vir(ms_hi, 1)}")
print(f"  P(pelo menos um silêncio de {MINUTOS} minutos em {HORAS} horas): {vir(p_tarde * 100, 1)}%")

# ── CAMADA 5 · até os cinco anos (conta de bolso) ──
# Premissa nossa, declarada: uma hora de conversa por dia com um adulto, dos 2 aos 5 anos.
HORAS_DIA, ANOS = 1, 3
explicacoes = EXPLICACOES_HORA * HORAS_DIA * 365 * ANOS
informacao = PERGUNTAS_HORA * HORAS_DIA * 365 * ANOS
print("\n" + "=" * 70)
print("  CAMADA 5 · ATÉ OS CINCO ANOS (conta de bolso)")
print("=" * 70)
print(f"\n  Premissa: {HORAS_DIA} hora de conversa por dia, dos 2 aos 5 anos ({ANOS} anos)")
print(f"  Pedidos de explicação: {EXPLICACOES_HORA} × 365 × {ANOS} = {mil(explicacoes)}")
print(f"  Perguntas de busca de informação: {PERGUNTAS_HORA} × 365 × {ANOS} = {mil(informacao)}")
print("  Harris (2015) chega a 'mais de 20 mil' pedidos de explicação antes dos cinco anos.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---
RODAPE = ('Ritmo: Chouinard (2007), Monographs of the SRCD 72(1), lido em fonte secundária · '
          'cálculo próprio | #365Probabilidades')

# ══ GRÁFICO 1 · o silêncio ══
fig1, ax1 = plt.subplots(figsize=(12, 8))
t = np.linspace(0, 10, 400)
ax1.fill_between(t, p_silencio(FAIXA_TOTAL[1], t) * 100, p_silencio(FAIXA_TOTAL[0], t) * 100,
                 color=CINZA, alpha=0.18, label='todas as perguntas (de 1 a 3 por minuto)')
ax1.plot(t, p_silencio(LAMBDA, t) * 100, color=VERMELHO, linewidth=3,
         label=f'perguntas de busca de informação ({PERGUNTAS_HORA} por hora)')
ax1.scatter([1, MINUTOS], [p_silencio(LAMBDA, 1) * 100, p5 * 100], color=DOURADO, s=90, zorder=5)
ax1.annotate(f'1 minuto: {vir(p_silencio(LAMBDA, 1) * 100, 0)}%', (1, p_silencio(LAMBDA, 1) * 100),
             xytext=(1.5, 45), fontsize=12, arrowprops=dict(arrowstyle='-', color=CINZA))
ax1.annotate(f'5 minutos: {vir(p5 * 100, 2)}%', (MINUTOS, p5 * 100),
             xytext=(5.6, 1.2), fontsize=12, arrowprops=dict(arrowstyle='-', color=CINZA))
ax1.set_yscale('log')
ax1.set_ylim(0.0005, 100)
ax1.set_yticks([0.001, 0.01, 0.1, 1, 10, 100])
ax1.set_yticklabels(['0,001%', '0,01%', '0,1%', '1%', '10%', '100%'])
ax1.set_xlabel('Minutos de conversa', fontsize=12)
ax1.set_ylabel('Chance de nenhuma pergunta (escala logarítmica)', fontsize=12)
ax1.set_title('A chance de silêncio cai depressa\n'
              f'P(nenhuma pergunta em t minutos) = e^(−λt), λ = {vir(LAMBDA, 2)} por minuto',
              fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=11, loc='lower left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-121-grafico-01-o-silencio.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · a espera pela próxima pergunta ══
fig2, ax2 = plt.subplots(figsize=(12, 8))
seg = np.linspace(0, 240, 500)
dens = stats.expon(scale=60 / LAMBDA).pdf(seg)
ax2.plot(seg, dens, color=VERDE, linewidth=3)
ax2.fill_between(seg, dens, where=seg <= mediana_seg, color=VERDE, alpha=0.25)
ax2.axvline(mediana_seg, color=DOURADO, linestyle='--', linewidth=2)
ax2.text(mediana_seg + 4, dens.max() * 0.85, f'metade das perguntas chega\nem até {vir(mediana_seg, 0)} segundos',
         fontsize=12, color=DOURADO)
ax2.axvline(p90_seg, color=CINZA, linestyle='--', linewidth=2)
ax2.text(p90_seg + 4, dens.max() * 0.45, f'90% chegam em até\n{vir(p90_seg, 0)} segundos', fontsize=12, color=CINZA)
ax2.set_xlabel('Segundos até a próxima pergunta', fontsize=12)
ax2.set_ylabel('Densidade', fontsize=12)
ax2.set_title('Quanto tempo até a próxima pergunta\n'
              f'distribuição exponencial, média de {vir(60 / LAMBDA, 0)} segundos', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-121-grafico-02-a-espera.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · as rajadas ══
fig3, ax3 = plt.subplots(figsize=(12, 8))
nomes = ['sem rajadas\n(Poisson)'] + [f'variação de\n{vir(100 / np.sqrt(k), 0)}% no ritmo' for k in FORMAS[1:]]
valores = [rajadas[k] * 100 for k in FORMAS]
barras = ax3.bar(nomes, valores, color=[VERMELHO] + [DOURADO] * (len(FORMAS) - 1), alpha=0.85, width=0.6)
for b, v in zip(barras, valores):
    ax3.text(b.get_x() + b.get_width() / 2, v + 0.3, f'{vir(v, 2 if v < 1 else 1)}%', ha='center',
             fontsize=13, fontweight='bold')
ax3.set_ylim(0, max(valores) * 1.2)
ax3.set_ylabel('Chance de cinco minutos sem pergunta (%)', fontsize=12)
ax3.set_title('Com rajadas, o silêncio fica mais provável\n'
              f'mesmo ritmo médio de {PERGUNTAS_HORA} perguntas por hora; o grau de rajada não é publicado',
              fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-121-grafico-03-as-rajadas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · o maior silêncio de uma tarde ══
fig4, ax4 = plt.subplots(figsize=(12, 8))
ax4.hist(maior_silencio, bins=np.arange(1.5, 9.01, 0.1), color=VERDE, alpha=0.8)
ax4.axvline(MINUTOS, color=VERMELHO, linewidth=2.5)
ax4.text(MINUTOS + 0.1, ax4.get_ylim()[1] * 0.9,
         f'cinco minutos\nacontece em {vir(p_tarde * 100, 0)}% das tardes', color=VERMELHO, fontsize=12)
ax4.axvline(ms_med, color=DOURADO, linewidth=2, linestyle='--')
ax4.annotate(f'mediana\n{vir(ms_med, 1)} minutos', (ms_med, ax4.get_ylim()[1] * 0.75),
             xytext=(1.6, ax4.get_ylim()[1] * 0.72), color=DOURADO, fontsize=12,
             arrowprops=dict(arrowstyle='-', color=DOURADO))
ax4.set_xlabel(f'Maior silêncio em {HORAS} horas de conversa (minutos)', fontsize=12)
ax4.set_ylabel('Sorteios', fontsize=12)
ax4.set_title(f'O maior silêncio de uma tarde de {HORAS} horas de conversa\n'
              f'{mil(SORTEIOS)} tardes simuladas, semente {SEMENTE}, sem rajadas', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-121-grafico-04-uma-tarde.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**A chance de uma criança passar cinco minutos de conversa sem fazer uma pergunta é de
0,18%.** Uma em 563.

A conta parte de um ritmo medido, de 76 perguntas de busca de informação por hora de
conversa com um adulto. Isso dá uma pergunta a cada 47 segundos, em média. Em cinco
minutos, o esperado são seis.

Metade das perguntas chega em até 33 segundos depois da anterior. Um minuto inteiro de
silêncio acontece em 28% das vezes. Dois minutos, em 8%.

Para ver um silêncio de cinco minutos, é preciso esperar em média 7,4 horas de conversa.
Numa tarde de três horas, ele aparece em uma de cada três tardes.

Esse número supõe perguntas chegando num ritmo constante, e criança pergunta em rajadas.
Com rajadas, o silêncio fica mais provável. Dependendo de quanto o ritmo varia ao longo
da conversa, a resposta vai de 0,18% a 13,6%. O grau de rajada não foi publicado, então
o 0,18% é o piso, e a resposta honesta é uma faixa.

Uma hora de conversa por dia, dos dois aos cinco anos, soma cerca de 27 mil pedidos de
explicação. Cada "por quê?" é um deles.

*Quando foi a última vez que você fez uma pergunta só para saber a resposta?*

---

### ⚠️ Limitações do Modelo

- **A monografia completa não foi lida.** O desenho dos estudos vem do resumo oficial.
  Os ritmos (76 por hora, de uma a três por minuto, 25 pedidos de explicação por hora)
  vêm de fontes secundárias que citam a monografia e concordam entre si. O total de
  perguntas e o número de horas transcritas não foram confirmados.
- **Quatro crianças.** O ritmo vem das transcrições de quatro crianças. Não há intervalo de confiança publicado para o
  ritmo, e por isso o notebook não desenha um. A faixa de uma a três perguntas por
  minuto é a única medida de variação disponível.
- **O ritmo é por hora de conversa com um adulto.** Não vale para a criança brincando
  sozinha, dormindo ou vendo televisão. O dia não diz quantas perguntas uma criança faz
  por dia.
- **O processo de Poisson supõe perguntas independentes e ritmo constante.** As duas
  premissas são falsas para crianças, e são usadas como referência. A camada 3 mostra
  que a resposta sobe até 13,6% quando o ritmo varia muito. O grau real de variação não
  é conhecido, e a mistura Gama-Poisson é uma escolha nossa.
- **O ritmo muda com a idade.** O número de 76 por hora é uma média do período de 1 a
  5 anos. Um bebê de um ano pergunta por gestos e sons, e a monografia trata esses
  pedidos em outro estudo.
- **A camada 5 é conta de bolso.** A hora diária de conversa é premissa nossa. Harris
  (2015) chega a "mais de 20 mil" pedidos de explicação antes dos cinco anos, com
  premissas próprias.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*